## Baseline Model

This model uses the best hyperparameter configuration identified in the **Hard-Coded Baseline** experiments  
(see `src/baseline/baseline-parameter-studing.ipynb`).  
It achieves a **Macro-F1 score of 0.728 on the public leaderboard**.

### Hyperparameters

The selected configuration lies in a stable performance plateau observed during tuning.

| Parameter    | Value |
|-------------|-------|
| WORD_NG_MAX | 2     |
| CHAR_NG_MAX | 5     |
| MIN_DF      | 2     |
| MAX_DF      | 0.85  |
| C_VALUE     | 1.0   |

### Design Choices

- **Timestamp** is excluded, as shown in `experiments/timestamp_is_already_cofied.ipynb`, since temporal information is already implicitly captured.
- **Source** is encoded using One-Hot Encoding (OHE) (see `experiments/why_hot_encoding_source.ipynb`).
- **Numerical features** are defined in `src/preprocessing_and_FE/building_base_dataset.ipynb`.
- The model operates on a unified **text** field (`title + article`) with basic markdown cleaning.

### Execution Environment

**Runtime:** 7 min 47 s  
**CPU:** 8 cores / 16 threads (AMD64)  
**RAM:** 31.33 GB  
**GPU:** none



In [1]:

# Libraries and Frameworks
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [ ]:
# Paths 

DEV_IN_PATH  = "../../data/processed/development_processed.csv"
EVAL_IN_PATH = "../../data/processed/evaluation_processed.csv"

SUB_OUT = "../../data/submission/submission_baseline.csv"




In [ ]:
# Parameters 
WORD_NG_MAX = 2
CHAR_NG_MAX = 5
MIN_DF      = 2
MAX_DF      = 0.85
C_VALUE     = 1

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]


In [4]:
# Load data and Features 

df_dev  = pd.read_csv(DEV_IN_PATH)
df_eval = pd.read_csv(EVAL_IN_PATH)

FEATURES = ["source", "text"] + NUM_COLS

X_dev  = df_dev[FEATURES]
y_dev  = df_dev["label"].astype(int)
X_eval = df_eval[FEATURES]


In [5]:
# Define Model (Lineaer Regression with TF-IDF and Scaling)

def make_model():
	pre = ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("w_tfidf", TfidfVectorizer(
				analyzer="word",
				ngram_range=(1, WORD_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=250_000
			), "text"),
			("c_tfidf", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, CHAR_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=300_000
			), "text"),
			("num", StandardScaler(), NUM_COLS),
		],
		remainder="drop",
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=C_VALUE,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	)

	return Pipeline([
		("pre", pre),
		("clf", clf),
	])

In [6]:
# Train and Predict

model = make_model()
model.fit(X_dev, y_dev)

pred = model.predict(X_eval)

In [8]:
# Submission

submission = pd.DataFrame({
	"Id": df_eval["Id"].astype(int),
	"Predicted": pred.astype(int)
})

submission.to_csv(SUB_OUT, index=False)
print("Saved baseline submission to:", SUB_OUT)

Saved baseline submission to: ../../data/submission/submission_baseline_1.csv
